# Natural Language Processing Lab - Assignment 02
## Task 2: WordNet Hypernyms & Ontological Hierarchy Traversal

**Objective**:
1. Select 10 English noun concepts and extract their immediate hypernyms (IS-A relationships).
2. Implement a recursive taxonomic path tracer from a synset up to the root entity (`entity.n.01`).
3. Analyze ontological depth and conceptual hierarchy across sample words.

In [ ]:
import nltk
import pandas as pd
from nltk.corpus import wordnet as wn

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Select 10 diverse noun words
selected_nouns = [
    "dog", "car", "apple", "chair", "oak",
    "doctor", "guitar", "river", "computer", "novel"
]

### 1. Extracting Primary Synsets and Direct Hypernyms

In [ ]:
noun_synset_map = {}
direct_hypernym_records = []

for noun in selected_nouns:
    # Select first noun synset
    syn = wn.synsets(noun, pos=wn.NOUN)[0]
    noun_synset_map[noun] = syn
    
    hypernyms = syn.hypernyms()
    hypernym_names = [h.name() for h in hypernyms] if hypernyms else ["None (Root)"]
    
    direct_hypernym_records.append({
        "Noun": noun,
        "Synset": syn.name(),
        "Direct Hypernym(s)": ", ".join(hypernym_names),
        "Definition": syn.definition()
    })

hypernym_table = pd.DataFrame(direct_hypernym_records)
display(hypernym_table)

### 2. Taxonomic Path Traversal to Root Node

In [ ]:
def trace_hypernym_lineage(synset):
    """Recursively builds path from the given synset up to entity.n.01"""
    path = [synset.name()]
    current = synset
    while current.hypernyms():
        current = current.hypernyms()[0]
        path.append(current.name())
    return path

# Trace lineage for 5 selected concepts
trace_sample = ["dog", "car", "apple", "doctor", "computer"]

for word in trace_sample:
    syn = noun_synset_map[word]
    lineage = trace_hypernym_lineage(syn)
    print(f"\n{'='*65}\nHIERARCHY CHAIN: {word.upper()} (Depth = {len(lineage)})\n{'='*65}")
    for depth, node in enumerate(reversed(lineage)):
        indent = "  " * depth
        arrow = "└── " if depth > 0 else ""
        print(f"{indent}{arrow}{node}")

### 3. Summary of Ontological Depths

In [ ]:
depth_records = []
for word, syn in noun_synset_map.items():
    lineage = trace_hypernym_lineage(syn)
    depth_records.append({
        "Concept": word,
        "Synset": syn.name(),
        "Taxonomic Depth": len(lineage),
        "Root Node": lineage[-1],
        "Full Lineage Path": " -> ".join(reversed(lineage))
    })

depth_df = pd.DataFrame(depth_records).sort_values(by="Taxonomic Depth", ascending=False).reset_index(drop=True)
display(depth_df[["Concept", "Synset", "Taxonomic Depth", "Root Node"]])